# 🩺 Hands-on Medical AI: Build an Interactive Chest X-Ray Diagnosis Web App with Gradio & Hugging Face

**Learning Objectives:**
1. Deploy a real-world pre-trained Medical Vision Transformer from **Hugging Face Hub** (`dima806/chest_xray_pneumonia_detection`).
2. Download and prepare benchmark Chest X-Ray (CXR) sample images.
3. Build an end-to-end inference pipeline using PyTorch & Hugging Face `transformers`.
4. Develop a full-featured, responsive web application in pure Python using **Gradio**.
5. Generate a public, shareable URL (`share=True`) directly from Google Colab to test on mobile devices and remote browsers.


In [ ]:
# 1. Install required packages
!pip install -q gradio transformers torch torchvision pillow

print("✅ Dependencies successfully installed!")


## 📁 Step 1: Download Sample Chest X-Ray Images

We will download two real pediatric chest radiographs (one **Normal** lung and one confirmed **Pneumonia** case) from open medical repositories (Wikimedia Commons / CDC) to serve as clickable example inputs in our web application.


In [ ]:
import os
import urllib.request
from PIL import Image

# Create samples directory
os.makedirs("samples", exist_ok=True)

# Sample chest radiograph URLs
samples = {
    "samples/normal.png": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Chest_Xray_PA_3-8-2010.png/500px-Chest_Xray_PA_3-8-2010.png",
    "samples/pneumonia.jpg": "https://upload.wikimedia.org/wikipedia/commons/0/01/Pneumonia_x_ray.jpg"
}

headers = {"User-Agent": "MedicalAIDemoBot/1.0 (contact: student@university.edu)"}

print("📥 Downloading sample radiographs...")
for filepath, url in samples.items():
    if not os.path.exists(filepath):
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req) as resp, open(filepath, "wb") as f:
            f.write(resp.read())
        print(f"   ✓ Downloaded: {filepath}")
    else:
        print(f"   ✓ Already exists: {filepath}")

print("\n🔍 Checking image properties:")
for filepath in samples.keys():
    img = Image.open(filepath)
    print(f"   - {filepath}: resolution {img.size}, format {img.format}, color mode {img.mode}")


## 🧠 Step 2: Load Pre-trained AI Model from Hugging Face Hub

We use the **`dima806/chest_xray_pneumonia_detection`** model:
* **Architecture:** Vision Transformer (ViT - fine-tuned from `google/vit-base-patch16-224-in21k`).
* **Task:** Binary classification on Chest X-Rays:
  * `NORMAL`: Clear lung fields without pathologic opacities.
  * `PNEUMONIA`: Parenchymal consolidation / infiltrates characteristic of bacterial or viral pneumonia.
* **Performance:** ~96% classification accuracy.
* We leverage the high-level `pipeline("image-classification")` for zero-boilerplate inference.


In [ ]:
import torch
from transformers import pipeline

# Automatically assign GPU if available in Colab runtime, else fallback to CPU
device = 0 if torch.cuda.is_available() else -1
print(f"Compute Device: {'CUDA GPU' if device == 0 else 'CPU'}")

# Initialize Hugging Face pipeline
model_id = "dima806/chest_xray_pneumonia_detection"
print(f"⏳ Loading '{model_id}' from Hugging Face Hub... (takes ~15-25 seconds)")

classifier = pipeline("image-classification", model=model_id, device=device)
print("✅ Model loaded successfully and ready for inference!")


## ⚙️ Step 3: Define the Prediction Function

Gradio connects to standard Python functions:
1. **Input:** An uploaded image provided as a `PIL.Image`.
2. **Process:** Model forward pass through the transformer pipeline.
3. **Output:** A dictionary mapping human-readable label names to confidence scores: `{"Label": float_probability}`. Gradio automatically renders this as a confidence bar chart.


In [ ]:
def predict_chest_xray(input_img):
    """
    Inference function for Gradio interface.
    Accepts a PIL Image and returns class confidence scores.
    """
    if input_img is None:
        return None

    # 1. Ensure 3-channel RGB format
    if input_img.mode != "RGB":
        input_img = input_img.convert("RGB")

    # 2. Run inference via pipeline
    predictions = classifier(input_img)

    # 3. Format clinical labels with visual indicators
    label_map = {
        "NORMAL": "✅ Normal (Healthy)",
        "PNEUMONIA": "⚠️ Pneumonia Detected"
    }

    # 4. Construct dictionary {label: probability} for gr.Label
    confidences = {}
    for item in predictions:
        raw_name = item["label"]
        display_name = label_map.get(raw_name, raw_name)
        confidences[display_name] = float(item["score"])

    return confidences

# Test the inference function on the downloaded pneumonia sample
from PIL import Image
print("🧪 Testing prediction function with 'samples/pneumonia.jpg':")
sample_output = predict_chest_xray(Image.open("samples/pneumonia.jpg"))
for label, prob in sample_output.items():
    print(f"   - {label}: {prob * 100:.2f}%")


In [ ]:
# ==============================================================================
# CELL 1 (FIXED): Grad-CAM Inference Function with Dynamic ViT Layer Detection
# ==============================================================================
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn

# Extract model and processor directly from our existing classifier pipeline
model = classifier.model
processor = classifier.image_processor
device = model.device


def get_vit_target_layer(model):
  """Dynamically locates the final transformer normalization layer across any

  Transformers version (supports both model.vit.layers and
  model.vit.encoder.layer).
  """
  # 1. New Transformers (v4.48+ / v5.0): model.vit.layers
  if (
      hasattr(model, "vit")
      and hasattr(model.vit, "layers")
      and len(model.vit.layers) > 0
  ):
    block = model.vit.layers[-1]
    return getattr(block, "layernorm_before", block)

  # 2. Classic Transformers: model.vit.encoder.layer
  if hasattr(model, "vit") and hasattr(model.vit, "encoder"):
    enc = model.vit.encoder
    layers = getattr(enc, "layers", getattr(enc, "layer", None))
    if layers and len(layers) > 0:
      return getattr(layers[-1], "layernorm_before", layers[-1])

  # 3. Universal Fallback: find the last LayerNorm before classifier in the model
  layernorms = [m for m in model.modules() if isinstance(m, nn.LayerNorm)]
  if layernorms:
    return layernorms[-1]

  raise RuntimeError("Could not automatically determine target layer for ViT.")


def overlay_cam(img_pil, cam_grid, alpha=0.45):
  """Resizes and blends a 14x14 CAM heatmap onto the original PIL image."""
  w, h = img_pil.size
  heatmap_resized = cv2.resize(cam_grid, (w, h), interpolation=cv2.INTER_CUBIC)
  heatmap_norm = np.uint8(255 * np.clip(heatmap_resized, 0, 1))

  # Apply JET colormap (Red = high activation, Blue = low activation)
  heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
  heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

  # Blend with original image
  orig_np = np.array(img_pil.convert("RGB"))
  blended = np.uint8(alpha * heatmap_color + (1 - alpha) * orig_np)
  return Image.fromarray(blended)


def predict_chest_xray_gradcam(input_img):
  """Enhanced inference function returning:

  1. Grad-CAM Heatmap overlaid on the original image (PIL.Image) 2. Diagnostic
  Title / Finding Summary (Markdown string) 3. Class Probabilities Dictionary
  (for gr.Label)
  """
  if input_img is None:
    return None, "### ℹ️ Please upload or select a radiograph", None

  # 1. Format input image to RGB
  if input_img.mode != "RGB":
    input_img = input_img.convert("RGB")

  # 2. Preprocess input for the model
  inputs = processor(images=input_img, return_tensors="pt").to(device)

  # Ensure parameters track gradients for backward pass
  for param in model.parameters():
    param.requires_grad = True

  # 3. Dynamically locate target layer and register hooks
  target_layer = get_vit_target_layer(model)

  activations = []
  gradients = []

  def forward_hook(module, inp, out):
    activations.append(out if isinstance(out, torch.Tensor) else out[0])

  def backward_hook(module, grad_in, grad_out):
    gradients.append(grad_out[0])

  h_fwd = target_layer.register_forward_hook(forward_hook)
  h_bwd = target_layer.register_full_backward_hook(backward_hook)

  # 4. Forward pass
  model.zero_grad()
  outputs = model(**inputs)
  logits = outputs.logits
  probs = torch.softmax(logits, dim=-1)

  # Predicted class index and confidence
  pred_idx = torch.argmax(probs, dim=-1).item()
  confidence = probs[0, pred_idx].item()
  target_score = logits[0, pred_idx]

  # 5. Backward pass to compute class-specific gradients
  target_score.backward()

  # Remove hooks
  h_fwd.remove()
  h_bwd.remove()

  # 6. Extract activations and gradients
  act = activations[0]
  grad = gradients[0]

  if act.dim() == 3:
    act = act[0]
  if grad.dim() == 3:
    grad = grad[0]

  # Skip CLS token (197 tokens -> keep 196 patch tokens for 14x14 grid)
  if act.shape[0] == 197:
    act = act[1:, :]
    grad = grad[1:, :]

  # Compute importance weights
  weights = grad.mean(dim=0)  # [768]
  cam = torch.matmul(act, weights)  # [196]
  cam = torch.relu(cam)  # Discard negative contributions

  cam_grid = cam.reshape(14, 14).detach().cpu().numpy()
  cam_norm = (cam_grid - cam_grid.min()) / (
      cam_grid.max() - cam_grid.min() + 1e-8
  )

  # 7. Create blended heatmap image
  cam_overlay_img = overlay_cam(input_img, cam_norm)

  # 8. Build diagnostic title and probability dictionary
  raw_label = model.config.id2label[pred_idx]
  if raw_label == "PNEUMONIA":
    title_md = (
        "### ⚠️ **Diagnostic Finding: PNEUMONIA DETECTED**"
        f" ({confidence * 100:.1f}% confidence)\n*High attention regions"
        " (red/yellow) indicate suspected pulmonary consolidation/infiltrates.*"
    )
  else:
    title_md = (
        "### ✅ **Diagnostic Finding: NORMAL / CLEAR LUNGS**"
        f" ({confidence * 100:.1f}% confidence)\n*Model focused on bilateral"
        " clear lung fields without focal infiltrates.*"
    )

  label_map = {
      "NORMAL": "✅ Normal (Healthy)",
      "PNEUMONIA": "⚠️ Pneumonia Detected",
  }
  confidences = {
      label_map.get(model.config.id2label[i], model.config.id2label[i]): float(
          probs[0, i].item()
      )
      for i in range(len(probs[0]))
  }

  return cam_overlay_img, title_md, confidences


# ==============================================================================
# Quick Test Run:
# ==============================================================================
print("🧪 Running Grad-CAM inference test on samples/pneumonia.jpg...")
test_cam_img, test_title, test_scores = predict_chest_xray_gradcam(
    Image.open("samples/pneumonia.jpg")
)
print("Diagnostic Title:", test_title)
print("Confidence Scores:", test_scores)
print(
    "✅ Grad-CAM Image generated successfully with resolution:",
    test_cam_img.size,
)


In [ ]:
# ==============================================================================
# CELL: Show Original Image & Grad-CAM Side-by-Side with Matplotlib
# ==============================================================================
from PIL import Image
import matplotlib.pyplot as plt


def show_gradcam_side_by_side(image_path):
  # 1. Load ảnh gốc và chạy hàm Grad-CAM
  orig_img = Image.open(image_path)
  cam_img, title_md, scores = predict_chest_xray_gradcam(orig_img)

  # 2. Tạo khung vẽ 1 hàng 2 cột
  fig, axes = plt.subplots(1, 2, figsize=(12, 6))

  # Ảnh 1: X-quang gốc
  axes[0].imshow(orig_img, cmap="gray")
  axes[0].set_title(
      "1. Original Chest X-Ray", fontsize=13, fontweight="bold", pad=10
  )
  axes[0].axis("off")

  # Ảnh 2: Grad-CAM Overlay
  axes[1].imshow(cam_img)
  axes[1].set_title(
      "2. Grad-CAM Saliency Map", fontsize=13, fontweight="bold", pad=10
  )
  axes[1].axis("off")

  # 3. Tiêu đề tổng quát (màu đỏ nếu Viêm phổi, xanh nếu Bình thường)
  top_label = max(scores, key=scores.get)
  top_prob = scores[top_label] * 100
  title_color = "#c0392b" if "Pneumonia" in top_label else "#27ae60"

  fig.suptitle(
      f"AI Diagnostic Result: {top_label} ({top_prob:.1f}%)",
      fontsize=16,
      fontweight="bold",
      color=title_color,
      y=0.98,
  )

  plt.tight_layout()
  plt.show()


# Chạy thử nghiệm ngay trên 2 ảnh mẫu
print("--- [TEST 1] PNEUMONIA CASE ---")
show_gradcam_side_by_side("samples/pneumonia.jpg")

print("--- [TEST 2] NORMAL CASE ---")
show_gradcam_side_by_side("samples/normal.png")


## 🎨 Step 4A: Quick Prototype with `gr.Interface` (5 Lines of Code)

`gr.Interface` is Gradio\'s high-level wrapper, perfect for quickly testing models and demonstrating concepts to students.


In [ ]:
import gradio as gr

# Build quick interface
quick_demo = gr.Interface(
    fn=predict_chest_xray,
    inputs=gr.Image(type="pil", label="Upload Chest X-Ray Radiograph"),
    outputs=gr.Label(num_top_classes=2, label="AI Diagnostic Confidence"),
    title="🩺 AI Chest X-Ray Pneumonia Screener",
    description="Upload a posteroanterior (PA) or anteroposterior (AP) chest X-ray to detect signs of pneumonia.",
    examples=[
        ["samples/normal.png"],
        ["samples/pneumonia.jpg"]
    ]
)

# Launch inline within notebook
quick_demo.launch(share=False)


## 🚀 Step 4B: Professional Web Application with `gr.Blocks`

When building capstone projects or research prototypes, `gr.Blocks` provides complete layout control:
* **Two-Column Layout:** (Left: Image uploader & Clickable examples; Right: Prediction confidence & Clinical notes).
* **Medical Disclaimer Banner:** Essential for any healthcare AI application.
* **Interactive Controls:** Explicit "Analyze" and "Clear" buttons.
* **Public Sharing (`share=True`):** Generates a public `https://xxxx.gradio.live` URL accessible from smartphones and external machines for 72 hours!


In [ ]:
import gradio as gr

with gr.Blocks(title="AI Chest X-Ray Diagnostic Assistant") as app:
    # 1. Header & Title
    gr.Markdown(
        """
        # 🩺 AI-Powered Chest Radiograph Diagnostic Assistant
        ### *Rapid screening for pulmonary pneumonia using Vision Transformer (ViT)*
        """
    )

    # 2. Medical Disclaimer Banner
    gr.HTML(
        """
        <div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 12px 16px; border-radius: 6px; margin-bottom: 20px; color: #856404; font-size: 14px;">
            <strong>⚠️ CLINICAL DISCLAIMER:</strong> This AI application is intended solely for <em>academic, research, and educational demonstration purposes</em>. It is NOT certified as a medical diagnostic device and must not replace professional clinical evaluation or radiologist consultation.
        </div>
        """
    )

    # 3. Main 2-Column Interface
    with gr.Row():
        # Left Column: Inputs & Examples
        with gr.Column(scale=1):
            gr.Markdown("### 1️⃣ Radiograph Input")
            image_input = gr.Image(
                type="pil",
                label="Chest X-Ray Image (PNG / JPEG)",
                sources=["upload", "clipboard"]
            )

            with gr.Row():
                clear_btn = gr.Button("🗑️ Reset", variant="secondary")
                predict_btn = gr.Button("🔍 Analyze Image", variant="primary")

            gr.Markdown("### 📂 Benchmark Sample Cases (Click to Test):")
            gr.Examples(
                examples=[
                    ["samples/normal.png"],
                    ["samples/pneumonia.jpg"]
                ],
                inputs=image_input,
                label="Pre-loaded Test Cases"
            )

        # Right Column: Predictions & Guidance
        with gr.Column(scale=1):
            gr.Markdown("### 2️⃣ Model Predictions & Confidence")
            output_label = gr.Label(
                num_top_classes=2,
                label="Predicted Class Probabilities"
            )

            gr.Markdown(
                """
                ---
                ### 📋 Radiographic Findings Reference:
                * **✅ Normal:** Uniform lung aeration, clear costophrenic angles, absence of focal consolidation or opacity.
                * **⚠️ Pneumonia:** Patchy airspace consolidation, peribronchial thickening, or diffuse alveolar infiltrates.
                * **Confidence Score (%):** Softmax posterior probability estimated by the Vision Transformer backbone.
                """
            )

    # 4. Event Binding
    predict_btn.click(
        fn=predict_chest_xray,
        inputs=image_input,
        outputs=output_label
    )

    # Real-time update when an example is clicked or a new image is dropped
    image_input.change(
        fn=predict_chest_xray,
        inputs=image_input,
        outputs=output_label
    )

    # Reset button handler
    clear_btn.click(
        fn=lambda: (None, None),
        inputs=None,
        outputs=[image_input, output_label]
    )

# 5. Launch the application
# Setting share=True creates a public Gradio link accessible over the internet
app.launch(share=True, debug=True)


# Advance

In [ ]:
# ==============================================================================
# CELL 2: Gradio Web App with Side-by-Side Grad-CAM Heatmap & Diagnosis
# ==============================================================================
import gradio as gr

with gr.Blocks(
    title="AI Chest Radiograph Screener with Grad-CAM XAI"
) as xai_demo:
  # Header
  gr.Markdown(
      """
        # 🩺 AI Chest Radiograph Screener with Explainable AI (Grad-CAM)
        ### *Vision Transformer (ViT) Diagnostic Assistant with Visual Saliency Localization*
        """
  )

  # Clinical Disclaimer
  gr.HTML(
      """
        <div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 12px 16px; border-radius: 6px; margin-bottom: 20px; color: #856404; font-size: 14px;">
            <strong>⚠️ CLINICAL DISCLAIMER:</strong> Intended solely for academic, research, and educational demonstration purposes. Saliency heatmaps represent model attention weights and must not replace radiologist evaluation.
        </div>
        """
  )

  with gr.Row():
    # Column 1: Input Radiograph
    with gr.Column(scale=1):
      gr.Markdown("### 1️⃣ Input Radiograph")
      input_img = gr.Image(
          type="pil",
          label="Upload Chest X-Ray",
          sources=["upload", "clipboard"],
      )
      with gr.Row():
        reset_btn = gr.Button("🗑️ Reset", variant="secondary")
        run_btn = gr.Button(
            "🔍 Analyze & Generate Grad-CAM", variant="primary"
        )

      gr.Markdown("### 📂 Benchmark Test Cases (Click to Test):")
      gr.Examples(
          examples=[["samples/normal.png"], ["samples/pneumonia.jpg"]],
          inputs=input_img,
          label="Pre-loaded Cases",
      )

    # Column 2: Grad-CAM Saliency Heatmap
    with gr.Column(scale=1):
      gr.Markdown("### 2️⃣ Grad-CAM Saliency Localization")
      cam_output = gr.Image(
          type="pil", label="Explainable AI Heatmap (JET Colormap)"
      )
      gr.Markdown(
          """
                * **🔥 Red / Yellow:** High attention (focal consolidations / infiltrates).
                * **❄️ Blue / Cyan:** Baseline activation (clear lung tissue / background).
                """
      )

    # Column 3: Diagnostic Title & Confidence Distribution
    with gr.Column(scale=1):
      gr.Markdown("### 3️⃣ Diagnostic Finding & Confidence")
      title_output = gr.Markdown("### ℹ️ Upload an image and click Analyze")
      label_output = gr.Label(num_top_classes=2, label="Class Probabilities")

  # Event Bindings
  run_btn.click(
      fn=predict_chest_xray_gradcam,
      inputs=input_img,
      outputs=[cam_output, title_output, label_output],
  )

  # Automatically update when user selects an example or uploads a new file
  input_img.change(
      fn=predict_chest_xray_gradcam,
      inputs=input_img,
      outputs=[cam_output, title_output, label_output],
  )

  # Reset button
  reset_btn.click(
      fn=lambda: (
          None,
          None,
          "### ℹ️ Upload an image and click Analyze",
          None,
      ),
      inputs=None,
      outputs=[input_img, cam_output, title_output, label_output],
  )

# Launch application with public shareable link
xai_demo.launch(share=True, debug=True)


## 💡 Student Exercises & Project Extensions

To take this tutorial further, challenge your students with the following advanced ideas:

1. **Explainable AI (XAI) with Saliency / Heatmaps:**
   * Integrate **Grad-CAM** or extract **Attention Rollout** from the Vision Transformer to highlight the anatomical regions of the lung influencing the pneumonia prediction.
2. **Clinical Decision Threshold Slider:**
   * Add a `gr.Slider(minimum=0.1, maximum=0.9, value=0.5, label="Detection Threshold")` allowing clinicians to adjust sensitivity vs. specificity (e.g., minimizing false negatives in screening).
3. **Multi-Class & Other Medical Modalities:**
   * Skin lesion / Melanoma classification (e.g., HAM10000 dataset).
   * Knee Osteoarthritis Kellgren-Lawrence (KL) severity grading.
   * Brain MRI tumor detection.
4. **Structured PDF Diagnostic Report Generation:**
   * Add a "Generate Medical Summary PDF" button that bundles patient ID, timestamp, the radiograph, and AI findings into a downloadable report.
